In [4]:
import numpy as np
import pandas as pd
import warnings

import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_style("ticks")
odx = pd.IndexSlice
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

%config InlineBackend.figure_format = 'retina'
%matplotlib inline

In [6]:
def read_url(link):
    """ Creates a pandas DataFrame from data online
    - Parameters:
        - link: link to the zipped data
    - Returns:
    """
    import io
    import requests
    import pandas as pd

    # Define URL and extract information
    response = requests.get(link)
    content = response.content
    # Convert into a Pandas DataFrame
    df = pd.read_csv(io.BytesIO(content), sep=',', compression='gzip')

    return df

reviews = read_url('https://data.insideairbnb.com/mexico/df/mexico-city/2025-06-25/data/reviews.csv.gz')
print(reviews.shape)

(1388226, 6)


In [7]:
reviews.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,10257549,59265221,2016-01-08,31043833,Karolis,The flat is very nice newly renovated. The hos...
1,10257549,59352234,2016-01-09,7248934,James,Benito and his wife were great guests. Perfect...
2,10257549,59456981,2016-01-10,52707457,Manuel,Fue algo express. No pude volar y contacté con...
3,10257549,59661086,2016-01-13,52989229,Annie,This was a very comfortable and conveniently l...
4,10257549,59806795,2016-01-15,51159033,Svenja,"Nice place, really close to te airport - Nice ..."


In [8]:
reviews[reviews['listing_id'] == 10257549]

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,10257549,59265221,2016-01-08,31043833,Karolis,The flat is very nice newly renovated. The hos...
1,10257549,59352234,2016-01-09,7248934,James,Benito and his wife were great guests. Perfect...
2,10257549,59456981,2016-01-10,52707457,Manuel,Fue algo express. No pude volar y contacté con...
3,10257549,59661086,2016-01-13,52989229,Annie,This was a very comfortable and conveniently l...
4,10257549,59806795,2016-01-15,51159033,Svenja,"Nice place, really close to te airport - Nice ..."
...,...,...,...,...,...,...
1358,10257549,1347325650887459677,2025-02-01,665917208,Fernanda Lucia,"Lugar muy tranquilo, con muy buena ubicación y..."
1359,10257549,1357403924139507317,2025-02-15,559873275,Daniel,Todo bien
1360,10257549,1377757479977791745,2025-03-15,473208679,Yuleimy Alhelí,"Benito es una persona muy educada y amable, el..."
1361,10257549,1382766816391027476,2025-03-22,322016426,Feer,"Mu has gracias a Benito por su atención, fue m..."


In [9]:
# añadimos numero de año_trimestre en que se realizo el comentario en una columna nueva
reviews['date'] = pd.to_datetime(reviews['date'])
# Finalmente, lo convertimos a entero para tener un valor numérico.
reviews['año_trimestre'] = (reviews['date'].dt.year.astype(str) + 
                       reviews['date'].dt.quarter.astype(str)).astype(int)

In [10]:
reviews.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments,año_trimestre
0,10257549,59265221,2016-01-08,31043833,Karolis,The flat is very nice newly renovated. The hos...,20161
1,10257549,59352234,2016-01-09,7248934,James,Benito and his wife were great guests. Perfect...,20161
2,10257549,59456981,2016-01-10,52707457,Manuel,Fue algo express. No pude volar y contacté con...,20161
3,10257549,59661086,2016-01-13,52989229,Annie,This was a very comfortable and conveniently l...,20161
4,10257549,59806795,2016-01-15,51159033,Svenja,"Nice place, really close to te airport - Nice ...",20161


In [11]:
# extract all non-null comments for the given listing
listing_id = 10257549
listing_comments = reviews.loc[reviews['listing_id'] == listing_id, 'comments'].dropna().reset_index(drop=True)
print(f"Found {len(listing_comments)} comments for listing {listing_id}")
listing_comments

Found 567 comments for listing 10257549


0      The flat is very nice newly renovated. The hos...
1      Benito and his wife were great guests. Perfect...
2      Fue algo express. No pude volar y contacté con...
3      This was a very comfortable and conveniently l...
4      Nice place, really close to te airport - Nice ...
                             ...                        
562    Lugar muy tranquilo, con muy buena ubicación y...
563                                            Todo bien
564    Benito es una persona muy educada y amable, el...
565    Mu has gracias a Benito por su atención, fue m...
566                                                👍👍👍👍👍
Name: comments, Length: 567, dtype: object

In [ ]:
# Agrupa por listing_id y año_trimestre y concatena los comentarios como "usuario:comentario"
def _combine_comments(df):
    tmp = df.loc[df['comments'].notna(), ['reviewer_name', 'reviewer_id', 'comments']].copy()
    if tmp.empty:
        return ''
    tmp['reviewer_name'] = tmp['reviewer_name'].fillna(tmp['reviewer_id'].astype(str))
    tmp['comments'] = tmp['comments'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    return ' || '.join(tmp['reviewer_name'].astype(str) + ':' + tmp['comments'])

grouped_comments = reviews.groupby(['listing_id', 'año_trimestre']).apply(_combine_comments).reset_index(name='all_comments')

# Ejemplo: mostrar las primeras filas
grouped_comments.head()

In [21]:
grouped_comments['all_comments'][0]

"Lindsay:Forget staying in a hotel. Stay at condesa haus. Fernando is super cool. The space/neighborhood is lovely. Don't forget to go to the roof. I wish I'd had more time to spend there. Till next time in Mexico city!"